## Column Generation example

In [1]:
using Pkg

Pkg.activate(".")
Pkg.instantiate()

  Activating project at `c:\Users\Patricia\Downloads\iniciacao cientifica\ESPPRC_route_generation`
┌ Warning: The active manifest file has dependencies that were resolved with a different julia version (1.10.9). Unexpected behavior may occur.
└ @ C:\Users\Patricia\Downloads\iniciacao cientifica\ESPPRC_route_generation\Manifest.toml:0


In [2]:
using HiGHS
using JuMP
using Graphs
using GraphPlot
using Plots
using LinearAlgebra
using Random

In [3]:
include("pdp.jl")

rota_da_solucao (generic function with 1 method)

## Problema aleatorios 

In [4]:
function problema_aleatorio(altura, largura, m, r, num_caminhoes, L)
    # gerando as cidades aleatorias 
    cid = Array{Float64}(undef, m, 2)
    for i = 1:m 
        cid[i, 1] = largura*rand()
        cid[i, 2] = altura*rand()
    end

    # calculando a distancia entre todas as cidades 
    C = Array{Float64}(undef, m, m)
    for i = 1:m
        C[i, i] = 0
    end
    for t = 1:m-1
        for s = t+1:m
            a = cid[t, 1]
            b = cid[t, 2]
            c = cid[s, 1]
            d = cid[s, 2]
            C[t, s] = sqrt((a-c)^2 + (b-d)^2)
            C[s,t]=C[t, s]
        end
    end

    # gerando uma quantidade r de tarefas 
    task = Array{Int64}(undef, r, 2)
    for i = 1:r
        a = rand(1:m)
        b = rand(1:m)
        while b == a 
            b = rand(1:m)
        end
        task[i, 1] = a
        task[i, 2] = b
    end

    # gerando uma quantidade r de janelas de tempo para realizacao de cada tarefa 
    W = Array{Float64}(undef, r, 2)
    for i = 1:r
        origem = task[i, 1]
        destino = task[i, 2]
        distancia = C[origem, destino]  # distância entre as cidades da tarefa
        c = distancia + (L - distancia) * rand()
        d = c + (L - c) * rand()     
        W[i, 1] = c
        W[i, 2] = d
    end
    return C, task, W
end

problema_aleatorio (generic function with 1 method)

## Grafo

In [5]:
function grafo(C, task, r)
    # Identificar apenas as cidades usadas nas tarefas
    usados = unique(vcat(task[:, 1], task[:, 2]))
    mapa_cidade = Dict(cidade => i for (i, cidade) in enumerate(usados))

    # Criar grafo apenas com as cidades usadas
    g = SimpleDiGraph(length(usados))

    # Criar matriz truncada inicializada com zeros
    C_truncada = zeros(Float64, length(usados), length(usados))

    # Preencher matriz apenas com distâncias das tarefas
    for i = 1:r
        origem_real = task[i, 1]
        destino_real = task[i, 2]
        origem = mapa_cidade[origem_real]
        destino = mapa_cidade[destino_real]
        C_truncada[origem, destino] = trunc(C[origem_real, destino_real])
    end

    # Criar lista de arestas com pesos
    weights = Dict()
    for i in 1:size(task, 1)
        origem_real, destino_real = task[i, 1], task[i, 2]
        origem = mapa_cidade[origem_real]
        destino = mapa_cidade[destino_real]
        if C_truncada[origem, destino] > 0
            add_edge!(g, origem, destino)
            weights[(origem, destino)] = C_truncada[origem, destino]
        end
    end

    # Obter rótulos das arestas
    graph_edges = collect(Graphs.edges(g))
    edge_labels = [weights[(u.src, u.dst)] for u in graph_edges]
    
    # Plotar o grafo (números originais das cidades no rótulo)
    #gplot(g, nodelabel=usados, edgelabel=edge_labels)
    display(gplot(g, nodelabel=usados, edgelabel=edge_labels))
    return g, weights
end

grafo (generic function with 1 method)

## Dados de entrada

In [6]:
# Representacao ficticia do Parana 
altura = 50
largura = 70

# m vai ser a quantidade de cidades dentro do Parana 
m = 10

# r vai ser a quantidade de tarefas que vou ter 
r = 6

# Numero de caminhoes
num_caminhoes = 3

# Limite de tempo
L = 200

# Semente
numero_primo = 	87213
teste = 2
semente = numero_primo + teste
Random.seed!(semente)

TaskLocalRNG()

In [7]:
C, task, W = problema_aleatorio(altura, largura, m, r, num_caminhoes, L)

([0.0 31.468866593768798 … 21.7907848765921 29.906505347448427; 31.468866593768798 0.0 … 9.73903843553628 24.973395789398815; … ; 21.7907848765921 9.73903843553628 … 0.0 21.368203002187876; 29.906505347448427 24.973395789398815 … 21.368203002187876 0.0], [5 8; 5 8; … ; 4 6; 8 4], [172.33171858992628 197.76735318951484; 149.07132140113302 157.97749511378362; … ; 96.25105939341316 132.7582522651925; 162.90004619568842 186.34585389301157])

In [8]:
A, solucao, g = A_final(C, task, W, num_caminhoes, r)

matriz task
matriz Wu
matriz W

Iteração 1

Solução ótima=[0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]
Valor ótimo=-3.0
Iterações=3
Base=[10, 8, 9, 4, 5, 6, 7]
Zj - Cj (Custo reduzido final) = [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Custo arestas:
	λ0 = -1.0
	λ = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
ESPPRC: L=198.3972846152315
	S = [0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 -1.0 -1.0 -1.0 -1.0 -1.0 0.0; 0.0 -1.0 0.0 -1.0 -1.0 -1.0 -1.0 0.0; 0.0 -1.0 -1.0 0.0 -1.0 -1.0 -1.0 0.0; 0.0 -1.0 -1.0 -1.0 0.0 -1.0 -1.0 0.0; 0.0 -1.0 -1.0 -1.0 -1.0 0.0 -1.0 0.0; 0.0 -1.0 -1.0 -1.0 -1.0 -1.0 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0]
	T = [396.794569230463 47.12654643751241 47.12654643751241 29.906505347448427 21.765189027039817 30.679979921348476 23.705797710213066 0.0; 396.794569230463 396.794569230463 94.25309287502482 53.355422288904 33.33631599690081 54.38577763156154 23.705797710213066 0.0; 396.794569230463 94.25309287502482 396.794569230463 53.355422

([1.0 0.0 … 1.0 1.0; 0.0 1.0 … 1.0 1.0; … ; 0.0 0.0 … 1.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.5, 1.0, 0.5, 0.5], 7)

In [9]:
 println("\nVerificando integralidade da solução...")

        eh_inteira = all(x -> abs(x - round(x)) < 1e-6, solucao)

        println("É inteira? ", eh_inteira)


Verificando integralidade da solução...
É inteira? false


In [10]:
rota_da_solucao(solucao, A, g)

6×5 Matrix{Float64}:
 0.0  0.0  0.0  1.0  1.0
 0.0  0.0  1.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  1.0
 0.0  1.0  0.0  1.0  0.0
 0.0  0.0  1.0  0.0  0.0